# Financial Fraud Detection: EDA on Credit Card Transactions
Exploratory data analysis (EDA) in Python for a fraud detection use case.

**Dataset:** `cc_data.csv` (credit card transactions with an `is_fraud` target).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## Task 1: Dimensions of the dataset

In [ ]:
data = pd.read_csv('cc_data.csv')   # update the path if needed
data = data.drop(columns='index', errors='ignore')
print('Dimensions of the dataset:', data.shape)

## Task 2: Unique values in categorical variables

In [ ]:
unique_values = data.select_dtypes(include=['object']).nunique()
print('Unique values in each categorical variable:')
print(unique_values)

## Task 3: Distribution of numerical variables

In [ ]:
data.describe()

## Task 4: Missing values

In [ ]:
missing_total = int(data.isnull().sum().sum())
print(data.isnull().sum())
data = data.dropna()

## Task 5: Summary statistics (mean, median, min, max)
The `50%` row is the median.

In [ ]:
num_cols = data.select_dtypes(include=np.number).drop(columns=['cc_num', 'zip'], errors='ignore')
num_cols.describe().T

## Task 6: Correlation between numerical variables

In [ ]:
corr = num_cols.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
pairs[pairs.abs() > 0.5].sort_values(ascending=False)

## Task 7: Distribution of `amt` across `is_fraud` categories

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='is_fraud', y='amt', data=data)
plt.yscale('log')
plt.title('Amount by fraud status (log scale)')
plt.show()

data.groupby('is_fraud')['amt'].agg(['count', 'mean', 'median', 'max'])

## Task 8: Outliers in `city_pop` and `amt`

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((s < low) | (s > high)).sum()
    return n, round(n / len(s) * 100, 2), low, high

for c in ['amt', 'city_pop']:
    n, pct, lo, hi = iqr_outliers(data[c])
    print(f'{c}: {n} outliers ({pct}%), bounds [{lo:.1f}, {hi:.1f}]')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=data['amt'], ax=ax[0])
sns.boxplot(x=data['city_pop'], ax=ax[1])
plt.show()

## Task 9: Trends over time

In [ ]:
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'], format='%d-%m-%Y %H:%M')
data = data.sort_values('trans_date_trans_time')

ts = data.set_index('trans_date_trans_time')
ts['amt'].resample('D').sum().plot(figsize=(14, 5), title='Daily total transaction amount')
plt.show()

data['hour'] = data['trans_date_trans_time'].dt.hour
data['weekday'] = data['trans_date_trans_time'].dt.day_name()
data.groupby('hour')['is_fraud'].mean().mul(100).plot(kind='bar', figsize=(12, 4), title='Fraud rate (%) by hour')
plt.show()

## Task 10: Distribution of the target across categories

In [ ]:
rate = (data.groupby('category')['is_fraud']
         .agg(total='count', frauds='sum', fraud_rate_pct=lambda s: round(s.mean() * 100, 2))
         .sort_values('fraud_rate_pct', ascending=False))
display(rate)
print('Overall fraud rate: %.2f%%' % (data['is_fraud'].mean() * 100))

## Task 11: Unusual or unexpected values

In [ ]:
for col in ['gender', 'state', 'category', 'job']:
    print(f"Unique values in '{col}':", data[col].nunique())
    print(data[col].value_counts().head())
    print('-' * 30)

## Task 12: Data entry errors or inconsistencies

In [ ]:
print('Duplicate rows:', data.duplicated().sum())
if 'trans_num' in data:
    print('Duplicate trans_num:', data['trans_num'].duplicated().sum())
print('amt <= 0:', (data['amt'] <= 0).sum())
print('lat out of range:', (~data['lat'].between(-90, 90)).sum(),
      '| long out of range:', (~data['long'].between(-180, 180)).sum())
print('Date range:', data['trans_date_trans_time'].min(), 'to', data['trans_date_trans_time'].max())
if 'dob' in data:
    age = (data['trans_date_trans_time'] - pd.to_datetime(data['dob'], errors='coerce')).dt.days / 365.25
    print('Age range:', round(age.min(), 1), '-', round(age.max(), 1))
print(data['gender'].value_counts())
print(data.dtypes)

## Task 13: Numerical variables across segments

In [ ]:
display(data.groupby('is_fraud')['amt'].describe())
display(data.groupby('category')['amt'].median().sort_values(ascending=False))

## Task 14: Top factors influencing `is_fraud`

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = pd.get_dummies(data[['amt', 'city_pop', 'category', 'gender', 'hour']], drop_first=True)
rf = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced',
                            random_state=42, n_jobs=-1).fit(X, data['is_fraud'])
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(10).plot(kind='barh', title='Top factors (Random Forest)').invert_yaxis()
plt.show()

## Task 15: Analysis report
The cell below builds the report from the results above. 

In [ ]:
from IPython.display import Markdown, display

n_rows, n_cols = data.shape
fraud_n = int(data['is_fraud'].sum())
fraud_pct = data['is_fraud'].mean() * 100
med = data.groupby('is_fraud')['amt'].median()
hourly = data.groupby('hour')['is_fraud'].mean() * 100
top_cat, top_cat_rate = rate.index[0], rate['fraud_rate_pct'].iloc[0]
tgt_corr = num_cols.corr()['is_fraud'].drop('is_fraud')
strong = pairs[pairs.abs() > 0.5]
amt_n, amt_pct, _, _ = iqr_outliers(data['amt'])
pop_n, pop_pct, _, _ = iqr_outliers(data['city_pop'])
dups = int(data.duplicated().sum())
top3 = ', '.join(imp.head(3).index)

report = f"""
### Analysis Report: EDA for Fraud Detection

1. **Dataset size:** {n_rows:,} rows and {n_cols} columns, covering {data['trans_date_trans_time'].min():%d %b %Y} to {data['trans_date_trans_time'].max():%d %b %Y}.
2. **Categorical variables:** {', '.join(f'{k} ({v})' for k, v in unique_values.items())}.
3. **Missing values:** {missing_total} missing values found; duplicate rows: {dups}.
4. **Class imbalance:** {fraud_n:,} fraudulent transactions ({fraud_pct:.2f}% of all). Accuracy alone is misleading for this data; use precision, recall, F1 and PR-AUC, and handle the imbalance (class weights, SMOTE or threshold tuning).
5. **Amount vs fraud:** median amount is {med.get(1, float('nan')):,.2f} for fraud vs {med.get(0, float('nan')):,.2f} for non-fraud.
6. **Correlations:** {len(strong)} numeric pairs with |r| > 0.5. Strongest link with `is_fraud`: `{tgt_corr.abs().idxmax()}` (r = {tgt_corr[tgt_corr.abs().idxmax()]:.3f}).
7. **Outliers (IQR rule):** `amt` has {amt_n:,} ({amt_pct}%) and `city_pop` has {pop_n:,} ({pop_pct}%).
8. **Fraud by category:** highest fraud rate is in `{top_cat}` ({top_cat_rate}%).
9. **Time pattern:** fraud rate peaks at hour {int(hourly.idxmax())}:00 ({hourly.max():.2f}%).
10. **Top predictive factors (Random Forest):** {top3}.

**Conclusion:** These findings show which transaction amounts, categories and times carry the most fraud risk. They guide feature engineering (hour, amount bands, category risk) and model choice for a fraud detection system.
"""
display(Markdown(report))